In [22]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import warnings
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import f1_score

from torch.utils.data import Dataset, DataLoader, random_split

path = kagglehub.dataset_download("shayanfazeli/heartbeat")
warnings.filterwarnings('ignore')
mitbih_train = pd.read_csv(path + '/mitbih_train.csv')
mitbih_test = pd.read_csv(path + '/mitbih_test.csv')

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [42]:
class hbDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe):
        self.data = torch.tensor(dataframe.to_numpy(),dtype=torch.float32)
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data = self.data[idx]
        return self.data[idx, :-1], self.data[idx, -1]

In [43]:
dataset = hbDataset(mitbih_train)
train_size = int(0.9 * len(dataset))
val_size = int(0.1 * len(dataset))
test_dataset = hbDataset(mitbih_test)
train_dataset, val_dataset = random_split(dataset, [0.9, 0.1])


In [44]:
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [45]:
def test(model, loader,loss_fn):
    loss_log = []
    acc_log = []
    model.eval()

    for data, target in loader:

        data, target = data.to(device),target.to(device)

        output = model(data)
        loss = loss_fn(output, target)
        loss_log.append(loss.item())

        _, predicted = torch.max(output, 1)
        acc = (predicted == target).sum() / target.size(0)
        acc_log.append(acc.item())

    return np.mean(loss_log), np.mean(acc_log)


def train_epoch(model, optimizer, train_loader,loss_fn):
    loss_log = []
    acc_log = []
    model.train()

    for data, target in train_loader:

        data, target = data.to(device),target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()
        loss_log.append(loss.item())

        _, predicted = torch.max(output, 1)
        acc = (predicted == target).sum() / target.size(0)
        acc_log.append(acc.item())

    return loss_log, acc_log


def train(model, optimizer, n_epochs, train_loader, val_loader, loss_fn, scheduler=None):
    train_loss_log, train_acc_log, val_loss_log, val_acc_log = [], [], [], []

    for epoch in range(n_epochs):
        train_loss, train_acc = train_epoch(model, optimizer, train_loader,loss_fn)
        val_loss, val_acc = test(model, val_loader,loss_fn)

        train_loss_log.extend(train_loss)
        train_acc_log.extend(train_acc)

        val_loss_log.append(val_loss)
        val_acc_log.append(val_acc)

        print(f"Epoch {epoch}")
        print(f" train loss: {np.mean(train_loss)}, train acc: {np.mean(train_acc)}")
        print(f" val loss: {val_loss}, val acc: {val_acc}\n")

        if scheduler is not None:
            scheduler.step()

    return train_loss_log, train_acc_log, val_loss_log, val_acc_log

In [46]:
class LinearModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(LinearModel, self).__init__()
        self.linear = nn.Linear(input_size, output_size)

    def forward(self, x):
        out = self.linear(x)
        return out

In [47]:
net = LinearModel(187,1)
net.to(device)
optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9)
train_loss_log, train_acc_log, val_loss_log, val_acc_log = train(
    net, optimizer, 20, train_loader, val_loader,loss_fn=nn.L1Loss()
)

Epoch 0
 train loss: 0.8938938919454813, train acc: 0.827731838474026
 val loss: 0.6760866705083499, val acc: 0.8258685774176661

Epoch 1
 train loss: 0.9077865094817305, train acc: 0.827958314062713
 val loss: 1.20702973726022, val acc: 0.8258685774176661

Epoch 2
 train loss: 0.8882169057796527, train acc: 0.8279130189352996
 val loss: 0.8111869205523582, val acc: 0.8258685774176661

Epoch 3
 train loss: 0.9242315504506424, train acc: 0.8279130189352996
 val loss: 0.7087443477480951, val acc: 0.8258685774176661

Epoch 4
 train loss: 0.9538308744403449, train acc: 0.8278224287288529
 val loss: 0.8111049363212864, val acc: 0.8258685774176661

Epoch 5
 train loss: 0.9193505073909636, train acc: 0.8280036091417461
 val loss: 0.7152287588067299, val acc: 0.8258685774176661

Epoch 6
 train loss: 0.9375262724888789, train acc: 0.8280036091417461
 val loss: 1.3784454660694094, val acc: 0.8258685774176661

Epoch 7
 train loss: 0.9193609355670678, train acc: 0.8278677238078861
 val loss: 2.113